# IVAP for Regression - Experimental results

In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Dict, Optional
import pandas as pd


artificial_datasets = ['linear_gaussian', 'nonlinear_sine', 'heteroscedastic',
                      'heavy_tailed', 'outliers', 'sparse_highdim', 'covariate_shift']
friedman_datasets = ['friedman1', 'friedman2', 'friedman3']

uci_datasets = ['airfoil', 'climate_bias', 'electricity']
other_datasets = ['star']

synthetic_datasets = artificial_datasets + friedman_datasets
real_datasets = uci_datasets + other_datasets

import warnings
warnings.filterwarnings("ignore")


## Read results

In [2]:
df_summary = pd.read_csv('output/all_synthetic_noise_3_10000.csv').iloc[:, 1:]

In [3]:
df_summary.head()

,scenario,model,mean,std,count
0,covariate_shift,Ridge,3.004853,0.063187,10
1,covariate_shift,LinearRegression,3.004854,0.063192,10
2,covariate_shift,LinearRegression CVAP - 1,3.110151,0.133667,10
3,covariate_shift,Ridge CVAP - 1,3.110160,0.133665,10
4,covariate_shift,Ridge CVAP - 10,3.139633,0.155678,10


## Summarise and export to Tex 

In [5]:
scenarios=(synthetic_datasets)
for sel in scenarios:
    df_summ = df_summary[df_summary.scenario==sel].copy()
    df_summ['calibration']='base'
    df_summ['base_model']=[i.split(' ')[0] for i in df_summ.iloc[:,1].values]
    df_summ.iloc[['CVAP' in row for row in df_summ['model'].values], -2] = [i.split('C')[1] for i in df_summ.iloc[:,1].values if len(i.split(' ')) > 1]
    print(sel)
    df_pivot = df_summ.pivot(index='base_model', columns='calibration', values='mean')
    average_values = df_pivot.select_dtypes(include=['number']).mean()
    df_pivot.loc['mean'] = average_values
    

linear_gaussian
nonlinear_sine
heteroscedastic
heavy_tailed
outliers
sparse_highdim
covariate_shift
friedman1
friedman2
friedman3


In [6]:
df_summ['base_model'].unique()

array(['RandomForest', 'GradientBoosting', 'ElasticNet', 'Lasso',
       'LinearRegression', 'Ridge', 'SVR(RBF)'], dtype=object)

In [7]:
df_summ = df_summary[df_summary.scenario==sel].copy()
df_summ['calibration']='base'
df_summ['base_model']=[i.split(' ')[0] for i in df_summ.iloc[:,1].values]

In [10]:
scen_desc = {}
scen_desc[scenarios[0]] = 'linear Gaussian'
scen_desc[scenarios[1]] = 'nonlinear'
scen_desc[scenarios[2]] = 'heteroscedastic noise'
scen_desc[scenarios[3]] = 'heavy-tailed noise'
scen_desc[scenarios[4]] = 'outlier contamination'
scen_desc[scenarios[5]] = 'sparse high-dimensional signals'
scen_desc[scenarios[6]] = 'covariate shift'
scen_desc[scenarios[7]] = 'Friedman 1'
scen_desc[scenarios[8]] = 'Friedman 2'
scen_desc[scenarios[9]] = 'Friedman 3'

In [11]:
scenario = scenarios[0]

for scenario in scenarios:
    df_summ = df_summary[df_summary.scenario==scenario].copy()
    df_summ['calibration']='base'
    df_summ['base_model']=[i.split(' ')[0] for i in df_summ.iloc[:,1].values]
    df_summ.iloc[['CVAP' in row for row in df_summ['model'].values], -2] = [i.split('C')[1] for i in df_summ.iloc[:,1].values if len(i.split(' ')) > 1]
    print(scenario)
    df_pivot = df_summ.pivot(index='base_model', columns='calibration', values='mean')
    df_pivot_std = df_summ.pivot(index='base_model', columns='calibration', values='std')
    average_values = df_pivot.select_dtypes(include=['number']).mean()
    df_pivot.loc['average'] = average_values
    df_pivot = df_pivot.iloc[:, [-1, 0, 1]]
    df_pivot.columns = ['none', 'CVAR1', 'CVAR10']
    df_pivot.index.name = 'base model'
    display(df_pivot.round(3))

    scratch = df_pivot.copy()
    df_s = scratch.style.format("{:.3f}")
    for row in scratch.index:
        col = scratch.loc[row].idxmin()
        df_s = df_s.format(lambda x: "\\textbf{" + f'{x:.3f}' + "}", subset=(row, col))

    tex_output = df_s.format_index(precision=0).to_latex(
        hrules=True, 
        column_format='l|l|rrr', 
        multicol_align='c',
        caption = 'RMSE  - ' + scen_desc[scenario] + ' dataset ')
    print(tex_output)


linear_gaussian


,none,CVAR1,CVAR10
base model,,,
ElasticNet,3.428,3.124,3.130
GradientBoosting,3.037,3.050,3.057
Lasso,3.574,3.324,3.328
LinearRegression,2.989,3.004,3.012
RandomForest,3.103,3.112,3.118
Ridge,2.989,3.004,3.012
SVR(RBF),3.063,3.077,3.082
average,3.169,3.099,3.106


\begin{table}
\caption{RMSE  - linear Gaussian dataset }
\begin{tabular}{l|l|rrr}
\toprule
 & none & CVAR1 & CVAR10 \\
base model &  &  &  \\
\midrule
ElasticNet & 3.428 & \textbf{3.124} & 3.130 \\
GradientBoosting & \textbf{3.037} & 3.050 & 3.057 \\
Lasso & 3.574 & \textbf{3.324} & 3.328 \\
LinearRegression & \textbf{2.989} & 3.004 & 3.012 \\
RandomForest & \textbf{3.103} & 3.112 & 3.118 \\
Ridge & \textbf{2.989} & 3.004 & 3.012 \\
SVR(RBF) & \textbf{3.063} & 3.077 & 3.082 \\
average & 3.169 & \textbf{3.099} & 3.106 \\
\bottomrule
\end{tabular}
\end{table}

nonlinear_sine


,none,CVAR1,CVAR10
base model,,,
ElasticNet,3.666,3.302,3.308
GradientBoosting,3.066,3.084,3.093
Lasso,3.796,3.506,3.511
LinearRegression,3.187,3.196,3.204
RandomForest,3.160,3.182,3.189
Ridge,3.187,3.196,3.204
SVR(RBF),3.117,3.138,3.145
average,3.311,3.229,3.236


\begin{table}
\caption{RMSE  - nonlinear dataset }
\begin{tabular}{l|l|rrr}
\toprule
 & none & CVAR1 & CVAR10 \\
base model &  &  &  \\
\midrule
ElasticNet & 3.666 & \textbf{3.302} & 3.308 \\
GradientBoosting & \textbf{3.066} & 3.084 & 3.093 \\
Lasso & 3.796 & \textbf{3.506} & 3.511 \\
LinearRegression & \textbf{3.187} & 3.196 & 3.204 \\
RandomForest & \textbf{3.160} & 3.182 & 3.189 \\
Ridge & \textbf{3.187} & 3.196 & 3.204 \\
SVR(RBF) & \textbf{3.117} & 3.138 & 3.145 \\
average & 3.311 & \textbf{3.229} & 3.236 \\
\bottomrule
\end{tabular}
\end{table}

heteroscedastic


...

\begin{table}
\caption{RMSE  - Friedman 1 dataset }
\begin{tabular}{l|l|rrr}
\toprule
 & none & CVAR1 & CVAR10 \\
base model &  &  &  \\
\midrule
ElasticNet & 4.382 & \textbf{3.892} & 3.899 \\
GradientBoosting & \textbf{3.133} & 3.175 & 3.190 \\
Lasso & 4.352 & \textbf{3.975} & 3.982 \\
LinearRegression & 3.887 & \textbf{3.879} & 3.886 \\
RandomForest & \textbf{3.272} & 3.309 & 3.323 \\
Ridge & 3.887 & \textbf{3.879} & 3.886 \\
SVR(RBF) & \textbf{3.232} & 3.266 & 3.279 \\
average & 3.735 & \textbf{3.625} & 3.635 \\
\bottomrule
\end{tabular}
\end{table}

friedman2


,none,CVAR1,CVAR10
base model,,,
ElasticNet,3.012,3.012,3.012
GradientBoosting,3.013,3.008,3.007
Lasso,3.012,3.012,3.012
LinearRegression,3.000,3.001,3.000
RandomForest,3.107,3.017,3.015
Ridge,3.000,3.001,3.000
SVR(RBF),3.012,3.009,3.007
average,3.022,3.009,3.008


\begin{table}
\caption{RMSE  - Friedman 2 dataset }
\begin{tabular}{l|l|rrr}
\toprule
 & none & CVAR1 & CVAR10 \\
base model &  &  &  \\
\midrule
ElasticNet & 3.012 & 3.012 & \textbf{3.012} \\
GradientBoosting & 3.013 & 3.008 & \textbf{3.007} \\
Lasso & 3.012 & 3.012 & \textbf{3.012} \\
LinearRegression & 3.000 & 3.001 & \textbf{3.000} \\
RandomForest & 3.107 & 3.017 & \textbf{3.015} \\
Ridge & 3.000 & 3.001 & \textbf{3.000} \\
SVR(RBF) & 3.012 & 3.009 & \textbf{3.007} \\
average & 3.022 & 3.009 & \textbf{3.008} \\
\bottomrule
\end{tabular}
\end{table}

friedman3


,none,CVAR1,CVAR10
base model,,,
ElasticNet,180.716,82.404,83.719
GradientBoosting,15.501,48.780,50.808
Lasso,136.909,82.453,83.770
LinearRegression,136.887,82.494,83.812
RandomForest,6.465,47.390,49.460
Ridge,136.887,82.494,83.812
SVR(RBF),139.039,103.783,103.930
average,107.486,75.686,77.044


\begin{table}
\caption{RMSE  - Friedman 3 dataset }
\begin{tabular}{l|l|rrr}
\toprule
 & none & CVAR1 & CVAR10 \\
base model &  &  &  \\
\midrule
ElasticNet & 180.716 & \textbf{82.404} & 83.719 \\
GradientBoosting & \textbf{15.501} & 48.780 & 50.808 \\
Lasso & 136.909 & \textbf{82.453} & 83.770 \\
LinearRegression & 136.887 & \textbf{82.494} & 83.812 \\
RandomForest & \textbf{6.465} & 47.390 & 49.460 \\
Ridge & 136.887 & \textbf{82.494} & 83.812 \\
SVR(RBF) & 139.039 & \textbf{103.783} & 103.930 \\
average & 107.486 & \textbf{75.686} & 77.044 \\
\bottomrule
\end{tabular}
\end{table}



In [104]:
# df_summary.to_csv('all_synthetic_noise_3.csv')